In [7]:
from interfacemaster.twinning_search import search_low_index_twinning
from interfacemaster.twinning_jobflow import TwinningJobflowMaker
from pymatgen.core.structure import Structure

stct = Structure.from_file('Li6PS5Cl.cif').get_primitive_structure()
# 1. 先搜索孪晶候选
search_results = search_low_index_twinning(parent = stct, 
                                           child = stct, 
                                           max_sigma=5, #最大sigma
                                           ortho_only=True, #要求正交超胞
                                          max_strain=1e-5,#最大strain（对于立方晶系直接设为很小）
                                          max_atoms=100, #最大原子数
                                           slab_length = 5, #slab厚度
                                          require_equivalent_terminations = True, #要求两个晶界端面等价
                                            termination_ftol=0.15, #端面分类精度
                                            termination_tol=0.15, #端面分类精度
                                          max_results = 2, #搜索到的符合要求的最大匹配数量
                                          prefilter_limit = 10, #搜索匹配的数量,
                                           debug_filters=True,
                                          )

--- 增强深度搜索开始 (Max Sigma: 5, HKL Limit: 4) ---
正在进行正交性过滤 (tol_ortho=0.01)...
正在进行原子数过滤 (max_atoms=100)...
  - atoms_est hkl [-3 -3 -2] sigma 3 -> 156
  - atoms_est hkl [-3 -1  0] sigma 3 -> 156
  - atoms_est hkl [-2  1  1] sigma 3 -> 156
  - atoms_est hkl [-1 -1 -1] sigma 3 -> 78
  - atoms_est hkl [-1 -1  0] sigma 3 -> 468
  - atoms_est hkl [-1  0  0] sigma 3 -> 78
  - atoms_est hkl [-1  0  0] sigma 3 -> 78
  - atoms_est hkl [-4 -3 -1] sigma 5 -> 260
  - atoms_est hkl [-3 -2 -1] sigma 5 -> 260
  - atoms_est hkl [-3 -2  1] sigma 5 -> 260
正在进行端面可生成性过滤 (require_equivalent_terminations=True)...
  - hkl [-1 -1 -1] sigma 3 -> term_pairs 1
  - hkl [-1  0  0] sigma 3 -> term_pairs 1
最终找到 2 个独特界面候选。
  轴/面: [-1 -1 -1] | Sigma: 3 | 角度: 60.0° | 类型: Crystal Axis/Normal
  轴/面: [-1  0  0] | Sigma: 3 | 角度: 60.0° | 类型: Crystal Axis/Normal


In [8]:
from sevenn.sevennet_calculator import SevenNetCalculator
calc = SevenNetCalculator(model = '/Users/jason/Desktop/LiNO2/checkpoint_sevennet_mf_ompa.pth',\
                          modal = 'omat24')
atoms = stct.to_ase_atoms()
atoms.calc = calc
bulk_energy_per_atom = atoms.get_potential_energy()/len(stct)

In [9]:
from jobflow import Flow
from jobflow.managers.local import run_locally
from interfacemaster.twinning_jobflow import TwinningJobflowMaker
from interfacemaster.twinning_search import search_low_index_twinning
from pymatgen.core.structure import Structure

maker = TwinningJobflowMaker(
    crystal_structure=stct,
    search_result=search_results[0],
    calc_type="sevennet",
    ml_model_path="/Users/jason/Desktop/LiNO2/checkpoint_sevennet_mf_ompa.pth",
    calc_kwargs={"modal": "omat24", "device": "cpu"},
    bulk_energy_per_atom = bulk_energy_per_atom,
    termination_ftol = 0.15,
    termination_tol = 0.15,
    trials = 10,
    slab_length = 5,
)

# 3) 生成 job 和 flow
job = maker.make()
flow = Flow([job])

# 4) 本地运行
response = run_locally(flow, create_folders = True)

2026-02-05 00:13:40,975 INFO Started executing jobs locally
2026-02-05 00:13:41,368 INFO Starting job - Twinning GB BO-Relax (Symmetric Terminations) (73f1a4a1-6a87-42c7-8991-7e8d6203d886)
共找到 11 对等价端面，开始逐个优化...
  - 优化 termination pair=(0.0441, 0.6875) (n_calls=10)
  - 优化 termination pair=(0.1009, 0.4097) (n_calls=10)
  - 优化 termination pair=(0.1009, 0.5441) (n_calls=10)
  - 优化 termination pair=(0.1875, 0.6009) (n_calls=10)
  - 优化 termination pair=(0.1875, 0.7789) (n_calls=10)
  - 优化 termination pair=(0.1875, 0.9097) (n_calls=10)
  - 优化 termination pair=(0.2789, 0.6009) (n_calls=10)
  - 优化 termination pair=(0.2789, 0.7789) (n_calls=10)
  - 优化 termination pair=(0.2789, 0.9097) (n_calls=10)
  - 优化 termination pair=(0.6009, 0.7789) (n_calls=10)
  - 优化 termination pair=(0.6009, 0.9097) (n_calls=10)
最低界面能: 0.4647 J/m^2，筛选 8 个端面进行结构优化...
  - 优化端面 1/8: dp1=0.1875, dp2=0.6009


/Users/jason/Documents/GitHub/interface_master/interfacemaster/twinning_jobflow.py:443: FutureWarning: Import ExpCellFilter from ase.filters
  ecf = ExpCellFilter(atoms, mask=[True, False, False, False, False, False])


  - 优化端面 2/8: dp1=0.1875, dp2=0.7789


KeyboardInterrupt: 